# 02 - K-Means Clustering

**Purpose**: Interactive K-Means clustering with real-time visualizations

**Features**:
- Adjustable K value with instant feedback
- Elbow method & silhouette analysis
- Live cluster visualizations
- Cluster profiling and naming
- PCA visualization

**Outputs**:
- Cluster assignments
- Cluster profiles
- Quality metrics
- Visualizations

---

## 1. Setup & Load Data

In [ ]:
# Import libraries
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score

from notebook_utils import (
    setup_notebook, display_dataframe_summary,
    plot_cluster_distribution, plot_feature_distributions,
    plot_cluster_profiles, save_results
)

%matplotlib inline
print("✓ Imports loaded")

In [ ]:
# Load state
cfg, state = setup_notebook(
    title="K-Means Clustering Analysis",
    market="germany"
)

# Load preprocessed data
df_latest = state.load('df_latest')
feature_cols = state.load('feature_cols')
static_features = state.load('static_features')
dynamic_features = state.load('dynamic_features')

if df_latest is None:
    raise ValueError("Data not found! Please run 01_Data_Preprocessing.ipynb first")

display_dataframe_summary(df_latest, "Latest Data")

## 2. Feature Selection & Configuration

**Choose which features to use for clustering:**

In [ ]:
# ============================================================================
# CLUSTERING CONFIGURATION - Adjust these parameters
# ============================================================================

# Feature set selection
USE_FEATURE_SET = 'combined'  # Options: 'static', 'dynamic', 'combined'

# K-Means parameters
OPTIMAL_K = 5                 # Number of clusters (will validate below)
RANDOM_STATE = 42             # For reproducibility
N_INIT = 10                   # Number of initializations
MAX_ITER = 300                # Maximum iterations

# Select features based on configuration
if USE_FEATURE_SET == 'static':
    clustering_features = static_features
elif USE_FEATURE_SET == 'dynamic':
    clustering_features = dynamic_features
else:  # combined
    clustering_features = feature_cols

print("📋 Clustering Configuration:")
print("=" * 80)
print(f"Feature Set:      {USE_FEATURE_SET}")
print(f"Features:         {len(clustering_features)}")
print(f"Optimal K:        {OPTIMAL_K}")
print(f"Random State:     {RANDOM_STATE}")
print("=" * 80)

## 3. Data Preparation & Scaling

In [ ]:
# Prepare data for clustering
df_cluster = df_latest[clustering_features].copy()

# Handle missing values
df_cluster = df_cluster.fillna(df_cluster.median())

print(f"\n📊 Clustering Dataset:")
print(f"  Samples:   {len(df_cluster):,}")
print(f"  Features:  {len(clustering_features)}")
print(f"  Missing:   {df_cluster.isnull().sum().sum()}")

In [ ]:
# Feature scaling (Z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print("\n✓ Features scaled using StandardScaler")
print(f"  Mean: {X_scaled.mean():.6f}")
print(f"  Std:  {X_scaled.std():.6f}")

## 4. Optimal K Selection

### 4.1 Elbow Method

In [ ]:
# Calculate inertia for different K values
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []

print("\n🔄 Testing different K values...\n")

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(X_scaled, labels))
    
    print(f"  K={k}: Inertia={kmeans.inertia_:.2f}, "
          f"Silhouette={silhouette_scores[-1]:.3f}, "
          f"Calinski-Harabasz={calinski_scores[-1]:.2f}")

print("\n✓ K-value testing complete")

In [ ]:
# Plot elbow curve and quality metrics
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Elbow curve
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(OPTIMAL_K, color='red', linestyle='--', alpha=0.7, label=f'Selected K={OPTIMAL_K}')
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Silhouette score
ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.axvline(OPTIMAL_K, color='red', linestyle='--', alpha=0.7, label=f'Selected K={OPTIMAL_K}')
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Calinski-Harabasz score
ax3.plot(K_range, calinski_scores, 'mo-', linewidth=2, markersize=8)
ax3.axvline(OPTIMAL_K, color='red', linestyle='--', alpha=0.7, label=f'Selected K={OPTIMAL_K}')
ax3.set_xlabel('Number of Clusters (K)', fontsize=12)
ax3.set_ylabel('Calinski-Harabasz Score', fontsize=12)
ax3.set_title('Calinski-Harabasz Index', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.show()

# Recommendations
best_silhouette_k = list(K_range)[np.argmax(silhouette_scores)]
best_calinski_k = list(K_range)[np.argmax(calinski_scores)]

print(f"\n📊 Recommendations:")
print(f"  Best Silhouette Score:        K={best_silhouette_k} (Score: {max(silhouette_scores):.3f})")
print(f"  Best Calinski-Harabasz Score: K={best_calinski_k} (Score: {max(calinski_scores):.2f})")
print(f"  Current Selection:            K={OPTIMAL_K}")

### 4.2 Detailed Silhouette Analysis

In [ ]:
# Create silhouette plot for selected K
def plot_silhouette_analysis(X, k, random_state=42):
    """Create silhouette plot for given K"""
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(X)
    
    silhouette_avg = silhouette_score(X, labels)
    sample_silhouette_values = silhouette_samples(X, labels)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    y_lower = 10
    
    for i in range(k):
        # Aggregate silhouette scores for samples in cluster i
        ith_cluster_silhouette_values = sample_silhouette_values[labels == i]
        ith_cluster_silhouette_values.sort()
        
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.nipy_spectral(float(i) / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, ith_cluster_silhouette_values,
                         facecolor=color, edgecolor=color, alpha=0.7)
        
        # Label clusters
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    ax.set_xlabel('Silhouette Coefficient', fontsize=12)
    ax.set_ylabel('Cluster Label', fontsize=12)
    ax.set_title(f'Silhouette Plot for K={k} (Avg Score: {silhouette_avg:.3f})',
                fontsize=14, fontweight='bold')
    
    # Average silhouette score line
    ax.axvline(x=silhouette_avg, color="red", linestyle="--", 
              label=f'Average Score: {silhouette_avg:.3f}')
    ax.legend()
    
    ax.set_yticks([])
    ax.set_xlim([-0.1, 1])
    plt.tight_layout()
    plt.show()
    
    return silhouette_avg

# Plot for selected K
sil_score = plot_silhouette_analysis(X_scaled, OPTIMAL_K, RANDOM_STATE)

## 5. Final K-Means Clustering

In [ ]:
# Train final model
print(f"\n🔄 Training K-Means with K={OPTIMAL_K}...\n")

kmeans_final = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    max_iter=MAX_ITER
)

cluster_labels = kmeans_final.fit_predict(X_scaled)

# Add cluster labels to dataframe
df_result = df_latest.copy()
df_result['cluster'] = cluster_labels

# Calculate metrics
final_inertia = kmeans_final.inertia_
final_silhouette = silhouette_score(X_scaled, cluster_labels)
final_calinski = calinski_harabasz_score(X_scaled, cluster_labels)

print("✓ K-Means clustering complete!\n")
print("📊 Final Metrics:")
print("=" * 80)
print(f"  Inertia:              {final_inertia:.2f}")
print(f"  Silhouette Score:     {final_silhouette:.3f}")
print(f"  Calinski-Harabasz:    {final_calinski:.2f}")
print(f"  Iterations:           {kmeans_final.n_iter_}")
print("=" * 80)

## 6. Cluster Distribution

In [ ]:
# Visualize cluster distribution
cluster_counts = plot_cluster_distribution(df_result, cluster_col='cluster', 
                                          title=f'K-Means Cluster Distribution (K={OPTIMAL_K})')

print("\n📊 Cluster Sizes:")
for cluster_id, count in cluster_counts.items():
    pct = count / len(df_result) * 100
    print(f"  Cluster {cluster_id}: {count:4d} companies ({pct:5.1f}%)")

## 7. Cluster Profiling

In [ ]:
# Calculate cluster profiles (mean values per cluster)
cluster_profiles = df_result.groupby('cluster')[clustering_features].mean()

print("\n📊 Cluster Profiles (Mean Values):")
display(cluster_profiles)

# Save profiles
state.save('kmeans_cluster_profiles', cluster_profiles)

In [ ]:
# Visualize cluster profiles as heatmap
plot_cluster_profiles(cluster_profiles, title=f'K-Means Cluster Profiles (K={OPTIMAL_K})')

In [ ]:
# Feature distributions across clusters (sample features)
sample_features = clustering_features[:6]
plot_feature_distributions(df_result, sample_features, cluster_col='cluster', max_features=6)

## 8. PCA Visualization

In [ ]:
# PCA for visualization
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

explained_var = pca_2d.explained_variance_ratio_
print(f"\n📊 PCA Explained Variance:")
print(f"  PC1: {explained_var[0]:.1%}")
print(f"  PC2: {explained_var[1]:.1%}")
print(f"  Total: {explained_var.sum():.1%}")

In [ ]:
# 2D PCA scatter plot
fig, ax = plt.subplots(figsize=(12, 8))

# Plot each cluster
colors = plt.cm.nipy_spectral(np.linspace(0, 1, OPTIMAL_K))

for cluster_id in range(OPTIMAL_K):
    mask = cluster_labels == cluster_id
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
              c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
              alpha=0.6, s=50, edgecolors='k', linewidth=0.5)

# Plot centroids
centroids_pca = pca_2d.transform(kmeans_final.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
          c='red', marker='X', s=300, edgecolors='black',
          linewidth=2, label='Centroids', zorder=10)

ax.set_xlabel(f'PC1 ({explained_var[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({explained_var[1]:.1%} variance)', fontsize=12)
ax.set_title(f'K-Means Clusters in PCA Space (K={OPTIMAL_K})', 
            fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA visualization
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3d = pca_3d.fit_transform(X_scaled)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

for cluster_id in range(OPTIMAL_K):
    mask = cluster_labels == cluster_id
    ax.scatter(X_pca_3d[mask, 0], X_pca_3d[mask, 1], X_pca_3d[mask, 2],
              c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
              alpha=0.6, s=50, edgecolors='k', linewidth=0.5)

# Plot centroids
centroids_3d = pca_3d.transform(kmeans_final.cluster_centers_)
ax.scatter(centroids_3d[:, 0], centroids_3d[:, 1], centroids_3d[:, 2],
          c='red', marker='X', s=300, edgecolors='black',
          linewidth=2, label='Centroids', zorder=10)

explained_3d = pca_3d.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({explained_3d[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({explained_3d[1]:.1%})', fontsize=11)
ax.set_zlabel(f'PC3 ({explained_3d[2]:.1%})', fontsize=11)
ax.set_title(f'K-Means Clusters in 3D PCA Space (K={OPTIMAL_K})\nTotal Variance: {explained_3d.sum():.1%}',
            fontsize=14, fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

## 9. Save Results

In [ ]:
# Save results to state
state.save('kmeans_model', kmeans_final)
state.save('kmeans_labels', cluster_labels)
state.save('kmeans_results', df_result)
state.save('kmeans_scaler', scaler)
state.save('kmeans_metrics', {
    'k': OPTIMAL_K,
    'inertia': final_inertia,
    'silhouette': final_silhouette,
    'calinski_harabasz': final_calinski,
    'iterations': kmeans_final.n_iter_
})

print("\n✓ K-Means results saved to state")

In [ ]:
# Save to files
output_dir = PROJECT_ROOT / 'output' / 'germany' / 'notebooks' / 'kmeans'

# Save cluster assignments
save_results(output_dir, df_result[['cluster']], prefix='cluster_assignments')

# Save cluster profiles
save_results(output_dir, cluster_profiles, prefix='cluster_profiles')

print(f"\n✓ Results saved to: {output_dir}")

## 10. Summary

In [ ]:
print("\n" + "=" * 80)
print("  ✓ K-MEANS CLUSTERING COMPLETE")
print("=" * 80)

print(f"\n📊 Summary:")
print(f"  Algorithm:            K-Means")
print(f"  Number of Clusters:   {OPTIMAL_K}")
print(f"  Feature Set:          {USE_FEATURE_SET}")
print(f"  Features Used:        {len(clustering_features)}")
print(f"  Companies:            {len(df_result):,}")

print(f"\n📈 Quality Metrics:")
print(f"  Silhouette Score:     {final_silhouette:.3f}")
print(f"  Calinski-Harabasz:    {final_calinski:.2f}")
print(f"  Inertia:              {final_inertia:.2f}")

print(f"\n📊 Cluster Distribution:")
for cluster_id, count in cluster_counts.items():
    pct = count / len(df_result) * 100
    print(f"  Cluster {cluster_id}: {count:4d} ({pct:5.1f}%)")

print("\n📝 Next Steps:")
print("  → Compare with: 03_Hierarchical_Clustering.ipynb")
print("  → Or try: 04_DBSCAN_Clustering.ipynb")
print("  → Or jump to: 05_Algorithm_Comparison.ipynb")
print("=" * 80)